In [1]:
%pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.6 MB/s eta 0:00:00


In [2]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from ultralytics import YOLO
import matplotlib.pyplot as plt
from collections import Counter

# 1. Load Model (Hanya dilakukan SATU KALI agar memori efisien)
model_path = '/content/yolov11_emosi.onnx'
try:
    model = YOLO(model_path)
    print("✅ Model YOLOv11 siap digunakan!")
    print("Silakan klik tombol di bawah untuk mulai menguji gambar.")
except Exception as e:
    print(f"Gagal memuat model. Error: {e}")

# 2. Buat Elemen UI (Tombol dan Layar Hasil)
upload_btn = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Foto Baru',
    button_style='info'
)
output_area = widgets.Output()

# 3. Fungsi Inti yang berjalan otomatis tiap kali tombol diklik
def on_upload_change(change):
    if not change['new']:
        return

    with output_area:
        clear_output(wait=True)

        uploaded_value = change['new']

        try:
            # Pengecekan otomatis versi ipywidgets (Mengatasi KeyError: 0)
            if isinstance(uploaded_value, dict):
                # Untuk ipywidgets versi lama (sering dijumpai di Colab)
                filename = list(uploaded_value.keys())[0]
                file_bytes = uploaded_value[filename]['content']
            else:
                # Untuk ipywidgets versi baru
                file_bytes = uploaded_value[0]['content']

            # Pastikan tipe data adalah bytes
            if not isinstance(file_bytes, bytes):
                file_bytes = bytes(file_bytes)

            # Konversi byte ke format array numpy
            nparr = np.frombuffer(file_bytes, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if img is None:
                print("❌ Gagal membaca gambar.")
                return

            # Jalankan inference YOLOv26
            results = model(img)

            # Plot gambar hasil deteksi
            res_plotted = results[0].plot()
            res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

            # Hitung Distribusi Emosi
            class_indices = results[0].boxes.cls.cpu().numpy()
            class_names = results[0].names
            detected_labels = [class_names[int(idx)] for idx in class_indices]
            distribution = Counter(detected_labels)

            # Tampilkan Gambar
            plt.figure(figsize=(10, 8))
            plt.imshow(res_rgb)
            plt.axis('off')
            plt.title("Hasil Deteksi YOLOv11", fontsize=16, fontweight='bold')
            plt.show()

            # Tampilkan Teks Distribusi
            print("-" * 35)
            print("📊 DISTRIBUSI HASIL DETEKSI:")
            print("-" * 35)

            if len(distribution) == 0:
                print("Tidak ada wajah yang terdeteksi.")
            else:
                total_wajah = sum(distribution.values())
                print(f"Total wajah terdeteksi: {total_wajah}\n")

                for label, count in distribution.items():
                    print(f" ▸ {label}: {count} wajah")
            print("-" * 35)

        except Exception as e:
            print(f"Terjadi kesalahan teknis: {e}")

        finally:
            # Kosongkan memori tombol agar selalu siap menerima file baru
            if isinstance(uploaded_value, dict):
                upload_btn.value.clear()
                upload_btn._counter = 0
            else:
                upload_btn.value = ()

# 4. Hubungkan fungsi ke tombol dan tampilkan UI
upload_btn.observe(on_upload_change, names='value')
display(upload_btn, output_area)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
✅ Model YOLOv11 siap digunakan!
Silakan klik tombol di bawah untuk mulai menguji gambar.


FileUpload(value={}, accept='image/*', button_style='info', description='Upload Foto Baru')

Output()